In [1]:
import sys, os
sys.path.insert(0, os.path.abspath('/home/ElasticNotebook'))
%load_ext ElasticNotebook
from elastic.core.common.pandas import compare_df
import pickle
import cudf

Enabled rmm statistics


In [2]:
%load_ext cudf.pandas

/opt/conda/envs/rapids-25.02/lib/python3.10/site-packages/cudf/pandas/__init__.py:65: UserWarning: cudf.pandas detected an already configured memory resource, ignoring 'CUDF_PANDAS_RMM_MODE'=managed_pool
  warnings.warn(


In [3]:
%LoadCheckpoint /home/dias-benchmarks/tpch/notebooks/q09/annotated/checkpoints/pre_cell_6.pickle

trying: ['load_supplier']
me:  6
trying: ['nation']
me:  4
trying: ['pd']
me:  0
trying: ['DATA_ROOT']
me:  0
trying: ['part']
me:  3
trying: ['STORAGE_OPTS']
me:  0
trying: ['orders']


me:  2
trying: ['load_orders']
me:  2
trying: ['load_nation']
me:  4
trying: ['load_partsupp']
me:  5
trying: ['load_part']
me:  3
trying: ['lineitem']


me:  1
trying: ['partsupp']
me:  5
trying: ['load_lineitem']
me:  1
trying: ['supplier']
me:  6
trying: ['tpch_parent']
me:  0


Declaring variable pd
Declaring variable DATA_ROOT
Declaring variable STORAGE_OPTS
Declaring variable tpch_parent
Declaring variable lineitem
Declaring variable load_lineitem
Declaring variable orders
Declaring variable load_orders
Declaring variable part
Declaring variable load_part
Declaring variable nation
Declaring variable load_nation
Declaring variable load_partsupp
Declaring variable partsupp
Declaring variable load_supplier
Declaring variable supplier


In [4]:
%%RecordEvent
%%time
### cell 6 ###

# Filter parts containing 'ghost'
fpart = part[part.P_NAME.str.contains("ghost")]

# Combine supplier and nation once (small tables)
sup_nat = supplier.merge(
    nation,
    left_on="S_NATIONKEY",
    right_on="N_NATIONKEY"
)

# Join lineitem → filtered parts → supplier+nation → partsupp
jn = (
    lineitem
    .merge(fpart, left_on="L_PARTKEY", right_on="P_PARTKEY")
    .merge(sup_nat, left_on="L_SUPPKEY", right_on="S_SUPPKEY")
    .merge(
        partsupp,
        left_on=["L_PARTKEY", "L_SUPPKEY"],
        right_on=["PS_PARTKEY", "PS_SUPPKEY"]
    )
)

# Compute profit contribution and discard unused columns before joining orders
tmp = (
    jn.assign(
        TMP=lambda df: df.L_EXTENDEDPRICE * (1 - df.L_DISCOUNT)
                      - df.PS_SUPPLYCOST * df.L_QUANTITY
    )
    [["N_NAME", "L_ORDERKEY", "TMP"]]
)

# Join orders (only needed cols), extract year, aggregate and sort
total = (
    tmp
    .merge(
        orders[["O_ORDERKEY", "O_ORDERDATE"]],
        left_on="L_ORDERKEY",
        right_on="O_ORDERKEY"
    )
    .assign(O_YEAR=lambda df: df.O_ORDERDATE.dt.year)
    .groupby(["N_NAME", "O_YEAR"], as_index=False, sort=False)["TMP"].sum()
    .sort_values(["N_NAME", "O_YEAR"], ascending=[True, False])
)

CPU times: user 90.8 ms, sys: 53.2 ms, total: 144 ms
Wall time: 155 ms


In [5]:
%Checkpoint /home/dias-benchmarks/tpch/notebooks/q09/rewritten/o4_mini_high_q09/checkpoints/post_cell_6_try_4.pickle

migration speed (bps): 803929966.5714841
---------------------------
variables to migrate:
part 48
load_lineitem 144
lineitem 48
DATA_ROOT 80
tpch_parent 54
load_nation 144
nation 48
fpart 48
sup_nat 48
total 48
load_partsupp 144
partsupp 48
STORAGE_OPTS 64
load_supplier 144
tmp 48
supplier 48
jn 48
pd 72
orders 48
load_orders 144
load_part 144
---------------------------
variables to recompute:
[]
---------------------------
cells to recompute:
[]
Checkpoint saved to: /home/dias-benchmarks/tpch/notebooks/q09/rewritten/o4_mini_high_q09/checkpoints/post_cell_6_try_4.pickle


In [6]:
%PrintCellInfo opt_cell_exec_info

======= Cell 0 =======
Input variables []
Active variables ['lineitem']
Intermediate variables []
Future variables []
Modified dataframes
======= Cell 1 =======
Input variables []
Active variables []
Intermediate variables ['orders']
Future variables ['lineitem']
Modified dataframes
======= Cell 2 =======
Input variables []
Active variables []
Intermediate variables ['part']
Future variables ['orders', 'lineitem']
Modified dataframes
======= Cell 3 =======
Input variables []
Active variables []
Intermediate variables ['nation']
Future variables ['orders', 'lineitem', 'part']
Modified dataframes
======= Cell 4 =======
Input variables []
Active variables []
Intermediate variables ['partsupp']
Future variables ['nation', 'orders', 'lineitem', 'part']
Modified dataframes
======= Cell 5 =======
Input variables []
Active variables []
Intermediate variables ['supplier']
Future variables ['lineitem', 'part', 'nation', 'orders', 'partsupp']
Modified dataframes
======= Cell 6 =======
Input varia

In [7]:

with open("/home/dias-benchmarks/tpch/notebooks/q09/opt_cell_exec_info_6_try_4.pkl", "wb") as f:
    pickle.dump(opt_cell_exec_info[6], f)


In [ ]:
opt_output = Out.get(4)

In [ ]:
total_opt = total
%LoadCheckpoint /home/dias-benchmarks/tpch/notebooks/q09/annotated/checkpoints/post_cell_6.pickle
assert compare_df(total_opt, total)

import numpy as np
import cudf
from elastic.core.common.pandas import is_type_styler
is_orig_output_pd = isinstance(orig_output, (pd.Series, pd.DataFrame, pd.Index))
is_opt_output_pd = isinstance(opt_output, (pd.Series, pd.DataFrame, pd.Index))
is_orig_output_array = isinstance(orig_output, (cudf.pandas._wrappers.numpy.ndarray, np.ndarray))
is_opt_output_array = isinstance(opt_output, (cudf.pandas._wrappers.numpy.ndarray, np.ndarray))
is_orig_output_styler = is_type_styler(type(orig_output))
is_opt_output_styler = is_type_styler(type(opt_output))
if is_orig_output_styler and is_opt_output_styler:
    assert orig_output.to_html() == opt_output.to_html()
elif is_orig_output_styler:
    assert orig_output.to_html() == opt_output.to_html()
elif is_opt_output_styler:
    assert opt_output.to_html() == orig_output

if is_orig_output_pd and is_opt_output_pd:
    assert orig_output.equals(opt_output)
# TODO(jie): this is a hack.
elif ((is_orig_output_pd or is_opt_output_pd) and (is_orig_output_array or is_opt_output_array)) or (is_orig_output_array and is_opt_output_array):
    assert list(orig_output) == list(opt_output)
else:
    assert orig_output == opt_output
